# minGRU Experiments
COMP6242 RNN's Revenge — Minimal GRU with parallel log-space scan (Feng et al. 2024)

Runs 9 experiments: 3 tasks × 3 lengths.

| Task | Lengths | Steps | Config key |
|------|---------|-------|------------|
| Shakespeare | 256 / 1024 / 2048 | 5K | `mingru_tinyshakespeare_{bs}` |
| Copy | short / medium / long | 5K | `mingru_longcopy_{len}` |
| Induction | short / medium / long | 20K (lr_decay 50K) | `mingru_induction_{len}` |

Protocol: dropout 0.05, AdamW (0.9/0.95), wd=0.1, lr 3e-4→3e-5, seed 42.

Note: minGRU's gate depends only on x_t (not h_{t-1}), so it cannot
selectively preserve memories through distractors. Induction/copy accuracy
is expected near random — this is an architectural limitation, not a bug.

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'bf16: {torch.cuda.is_bf16_supported()}')

In [ ]:
%cd /content/DL-RNNs-Revenge/minGRU

## Data Setup
Symlink to shared data generated by the transformer notebook (or generate fresh).

In [ ]:
import os
if not os.path.exists('data'):
    # Use data generated by transformer notebook
    if os.path.exists('/content/DL-RNNs-Revenge/transformer_project/data'):
        os.symlink('/content/DL-RNNs-Revenge/transformer_project/data', 'data')
    else:
        # Generate fresh if transformer hasn't run yet
        os.makedirs('data', exist_ok=True)
        os.chdir('/content/DL-RNNs-Revenge/transformer_project')
        os.system('python data/tinyshakespeare/prepare.py')
        os.system('python generate_longrange_copy.py')
        os.system('python generate_induction.py')
        os.chdir('/content/DL-RNNs-Revenge/minGRU')
        os.symlink('/content/DL-RNNs-Revenge/transformer_project/data', 'data')
print('Data contents:', os.listdir('data'))

## Shakespeare × 3

In [ ]:
!python scripts/train.py --exp mingru_tinyshakespeare_256
!python scripts/train.py --exp mingru_tinyshakespeare_1024
!python scripts/train.py --exp mingru_tinyshakespeare_2048

## Long-range Copy × 3

In [ ]:
!python scripts/train.py --exp mingru_longcopy_short
!python scripts/train.py --exp mingru_longcopy_medium
!python scripts/train.py --exp mingru_longcopy_long

## Induction × 3 (20K steps, lr_decay 50K)

In [ ]:
!python scripts/train.py --exp mingru_induction_short
!python scripts/train.py --exp mingru_induction_medium
!python scripts/train.py --exp mingru_induction_long

## Collect Results

In [ ]:
import json, glob, csv

print(f"{'Run':<45} {'Val PPL':>10} {'Masked PPL':>12} {'Masked Acc':>10}")
print('-' * 80)
for p in sorted(glob.glob('runs/*/eval_log.csv')):
    with open(p) as f:
        rows = list(csv.DictReader(f))
    if rows:
        best = min(rows, key=lambda r: float(r['val_loss']))
        name = p.split('/')[1]
        print(f"{name:<45} {float(best['val_ppl']):>10.4f} "
              f"{float(best['masked_ppl']):>12.4f} {float(best['masked_acc']):>10.4f}")